In [7]:
# S.0 Import thư viện
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, element_at
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import os, sys, glob, time, shutil

In [8]:
# S.0 Khởi tạo Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
assert sys.version_info < (3, 12), (
    f"PySpark 3.5.1 chỉ chạy Python <= 3.11 (đang dùng {sys.version.split()[0]}). "
    "Hãy chọn interpreter/kernel Python 3.11.")

spark = SparkSession.builder.appName("AirbnbSuperhostStream").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [9]:
# S.0 Đường dẫn model
MODEL_PATH = "streaming/models/rf_superhost"
DATA_CSV = "../data/Listings.csv"
# MODEL_PATH = "hdfs://master-node:9000/AirbnbData/models/rf_superhost"
# DATA_CSV = "hdfs://master-node:9000/AirbnbData/Listings.csv"

SOURCE_DIR = "streaming/stream_source"   # nguồn đã chia thành file nhỏ
INPUT_DIR = "streaming/stream_input"     # thư mục Spark theo dõi luồng
os.makedirs(SOURCE_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)

In [10]:
# S.1 Cột giữ lại cho luồng
KEEP = [
    "listing_id", "host_id", "city", "host_is_superhost",
    "price", "host_total_listings_count", "minimum_nights",
    "review_scores_rating", "review_scores_cleanliness", "review_scores_communication",
    "host_response_time", "instant_bookable", "host_identity_verified", "room_type",
]

In [11]:
# S.2 Chuẩn bị nguồn streaming (chạy 1 lần; có _SUCCESS nghĩa là đã ghi xong -> bỏ qua)
# Không inferSchema để khỏi quét toàn bộ file lớn -- readStream ở S.5 đã ép kiểu bằng SCHEMA.
if os.path.exists(SOURCE_DIR + "/_SUCCESS"):
    print("Đã có sẵn nguồn streaming -> bỏ qua S.2.")
else:
    raw = spark.read.csv(DATA_CSV, header=True, multiLine=True, escape='"', quote='"')
    mau = (raw.select(*KEEP)
              .dropna(subset=["host_is_superhost", "review_scores_rating"])
              .sample(False, 0.02, seed=42)   # lấy mẫu để có nhiều thành phố
              .limit(400))
    mau.repartition(20).write.option("header", True).mode("overwrite").csv(SOURCE_DIR)
    print("Đã chuẩn bị nguồn streaming (~400 listing, 20 file) trong:", SOURCE_DIR)

Đã có sẵn nguồn streaming -> bỏ qua S.2.


In [12]:
# S.3 Schema cho luồng (kiểu dữ liệu từng cột)
SCHEMA = StructType([
    StructField("listing_id", IntegerType(), True),
    StructField("host_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("host_is_superhost", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("host_total_listings_count", DoubleType(), True),
    StructField("minimum_nights", DoubleType(), True),
    StructField("review_scores_rating", DoubleType(), True),
    StructField("review_scores_cleanliness", DoubleType(), True),
    StructField("review_scores_communication", DoubleType(), True),
    StructField("host_response_time", StringType(), True),
    StructField("instant_bookable", StringType(), True),
    StructField("host_identity_verified", StringType(), True),
    StructField("room_type", StringType(), True),
])

In [13]:
# S.4 Nạp mô hình đã train
model = PipelineModel.load(MODEL_PATH)

In [14]:
# S.5 Khai báo nguồn đọc luồng (mỗi trigger đọc 1 file)
stream = (spark.readStream
          .schema(SCHEMA)
          .option("header", True)
          .option("maxFilesPerTrigger", 1)
          .csv(INPUT_DIR))

In [15]:
# S.6 Áp mô hình + lấy xác suất Superhost (phần tử thứ 2 của vector probability)
pred = model.transform(stream)
pred = pred.withColumn("p_superhost", element_at(vector_to_array("probability"), 2))
ket_qua = pred.select(
    "listing_id", "host_id", "city",
    col("host_is_superhost").alias("thuc_te"),
    col("prediction").alias("du_doan"),
    "p_superhost",
)

In [16]:
# S.7 Bật luồng, ghi kết quả vào bảng nhớ "ket_qua_stream"
query = (ket_qua.writeStream
         .format("memory").queryName("ket_qua_stream")
         .outputMode("append").start())
print("Đã bật luồng. Chạy S.8 để đổ dữ liệu, rồi xem S.9-S.12.")

Đã bật luồng. Chạy S.8 để đổ dữ liệu, rồi xem S.9-S.12.


In [17]:
# S.8 Đổ từng file vào thư mục theo dõi (3 giây/file) để mô phỏng luồng
for f in glob.glob(INPUT_DIR + "/*.csv"):
    os.remove(f)
files = sorted(glob.glob(SOURCE_DIR + "/part-*.csv"))
print("Bắt đầu đổ", len(files), "file vào luồng (3 giây/file)...")
for i, f in enumerate(files, 1):
    shutil.copy(f, INPUT_DIR + f"/batch_{i}.csv")
    print(f"  ({i}/{len(files)}) batch_{i}.csv")
    time.sleep(3)
print("Đã đổ xong. Chờ vài giây cho Spark xử lý nốt rồi xem S.9.")

Bắt đầu đổ 20 file vào luồng (3 giây/file)...
  (1/20) batch_1.csv
  (2/20) batch_2.csv
  (3/20) batch_3.csv
  (4/20) batch_4.csv
  (5/20) batch_5.csv
  (6/20) batch_6.csv
  (7/20) batch_7.csv
  (8/20) batch_8.csv
  (9/20) batch_9.csv
  (10/20) batch_10.csv
  (11/20) batch_11.csv
  (12/20) batch_12.csv
  (13/20) batch_13.csv
  (14/20) batch_14.csv
  (15/20) batch_15.csv
  (16/20) batch_16.csv
  (17/20) batch_17.csv
  (18/20) batch_18.csv
  (19/20) batch_19.csv
  (20/20) batch_20.csv
Đã đổ xong. Chờ vài giây cho Spark xử lý nốt rồi xem S.9.


In [18]:
# S.9 Kết quả real-time: số listing dự đoán theo thành phố
spark.sql("""
    SELECT city, du_doan, COUNT(*) AS so_luong
    FROM ket_qua_stream
    GROUP BY city, du_doan
    ORDER BY city, du_doan
""").show(50, truncate=False)

tong = spark.sql("SELECT COUNT(*) AS c FROM ket_qua_stream").collect()[0]["c"]
print("Tổng số listing đã chấm qua luồng:", tong)

+--------------+-------+--------+
|city          |du_doan|so_luong|
+--------------+-------+--------+
|Bangkok       |0.0    |1       |
|Bangkok       |1.0    |1       |
|Istanbul      |0.0    |1       |
|Mexico City   |0.0    |1       |
|Mexico City   |1.0    |1       |
|New York      |0.0    |12      |
|New York      |1.0    |4       |
|Paris         |0.0    |51      |
|Paris         |1.0    |10      |
|Rio de Janeiro|0.0    |3       |
|Rio de Janeiro|1.0    |1       |
|Rome          |0.0    |4       |
|Sydney        |0.0    |9       |
|Sydney        |1.0    |1       |
+--------------+-------+--------+

Tổng số listing đã chấm qua luồng: 100
